# Phoenix Phase-1 hardening: reproducible platform demo

This publication-facing notebook runs one declarative heterogeneous scene through full-wave synthetic acquisition, controlled complex noise, receiver-position mismatch, DAS, Born, DBIM, CSI, unified evaluation, and automatic reporting.

## 1. Platform flow

The notebook contains orchestration only. Physics and inverse algorithms remain in the mwisim package: YAML scene → CompositeCirclePhantom → SyntheticDataSource → Phase1Pipeline → DAS/Born/DBIM/CSI → metrics → report.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd().resolve()
if not (repo / 'mwisim').exists() and (repo.parent / 'mwisim').exists():
    repo = repo.parent
if not (repo / 'mwisim').exists():
    raise RuntimeError('Run this notebook from the phoenix-mwi repository or notebooks directory')
sys.path.insert(0, str(repo))

from mwisim.config import load_yaml_spec
from mwisim.pipeline import Phase1Pipeline


## 2. Inspect the declarative experiment

The configuration is data, not Python code. It records geometry, material values, acquisition dimensions, corruption levels, algorithm hyperparameters, evaluation threshold, and seed.

In [ ]:
config_path = repo / 'examples' / 'phase1_hardening.yaml'
spec = load_yaml_spec(config_path)
spec


## 3. Run the complete pipeline

The generated receiver data use perturbed true positions, while imaging and inversion use nominal positions. This deliberately creates model mismatch. Noise is complex Gaussian and scaled to the requested finite-vector SNR.

In [ ]:
output_dir = repo / 'docs' / 'notebook_phase1_output'
result = Phase1Pipeline(config_path).run(output_dir=output_dir)
result['problem_summary'], result['acceptance']


In [ ]:
print('method     chi-rel-L2      SSIM      data-residual')
for key in ('born', 'dbim', 'csi'):
    record = result['methods'][key]
    metric = record['metrics']
    print(f"{record['name']:<8}   {metric['rel_l2']:>10.4f}   {metric['ssim']:>7.4f}   {metric['data_residual']:>13.4e}")


## 4. Display reproducible artifacts

The same PNG and Markdown files are generated by the command-line driver, so notebook and CI results use one implementation.

In [ ]:
from IPython.display import Image, display

display(Image(filename=result['artifacts']['method_figure']))
display(Image(filename=result['artifacts']['residual_figure']))
print(result['artifacts']['report_md'])


## 5. Interpretation and claim boundary

A lower residual under noise and geometry mismatch is useful but does not prove a clinically correct permittivity map. Interpret relative L2, SSIM, support IoU, component count, localization, contrast recovery, and the common full-forward residual together. These remain 2-D synthetic software-validation experiments; measured-data and clinical claims belong to Phase 2 and later.